In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# just for pipeline
cols_to_keep = [
    'waypoint', 'flight_id', 'typecode', 'timestamp', 'time', 'latitude',
    'longitude', 'fl', 'altitude_ft', 'pressure', 'true_airspeed', #'mach',
    'aircraft_mass', 'fuel_flow',
    'day', 'fir', 'fl_diff',
    'deviation', 'is_excluded'
]

In [ ]:
# import
path_to_data = "/path/to/data"
metaf = pd.read_parquet(f"{path_to_data}/filed/filed_meta.parquet")
metao = pd.read_parquet(f"{path_to_data}/optimised/optimised_meta.parquet")
# filter just for pipeline
dff = pd.read_parquet(f"{path_to_data}/filed/filed_trajectories_EAGWP100.parquet")[cols_to_keep]
dfo = pd.read_parquet(f"{path_to_data}/optimised/optimised_trajectories_EAGWP100.parquet")[cols_to_keep]

In [ ]:
# df to Flight
from cane.utils import df_to_flight

fleetf = {
    fid: df_to_flight(group)
    for fid, group in dff.groupby("flight_id")
}
fleeto = {
    fid: df_to_flight(group)
    for fid, group in dfo.groupby("flight_id")
}

In [ ]:
# run aircraft performance
from cane.models.aircraft_performance import compute_aircraft_performance

fleetf = {fid: compute_aircraft_performance(flight, None) for fid, flight in fleetf.items()}
fleeto = {fid: compute_aircraft_performance(flight, None) for fid, flight in fleeto.items()}

In [ ]:
# update dataframes
dff = pd.concat([f.dataframe for f in list(fleetf.values())], ignore_index=True)
dfo = pd.concat([f.dataframe for f in list(fleeto.values())], ignore_index=True)

In [ ]:
# Create folder to save processed files
import os

path_to_processed = os.path.expanduser(f"{path_to_data}/processed")
os.makedirs(path_to_processed, exist_ok=True)

In [ ]:
# save to get ready for aCCF intersection and CoCiP on DelftBlue (HPC)
from cane.labels import time_bounds

for i, tb in enumerate(time_bounds):
    day = i + 1
    dff[dff["timestamp"].between(tb[0], tb[1])].reset_index(drop=True).to_parquet(f"{path_to_data}/processed/filed_trajectories_day{day}.parquet")
    dfo[dfo["timestamp"].between(tb[0], tb[1])].reset_index(drop=True).to_parquet(f"{path_to_data}/processed/optimised_trajectories_day{day}.parquet")

In [ ]:
# Now it's HPC time!
# download ERA5 data there for the relevant timebounds (either create symbolic link under .cache/pycontrails that points to era5 folder, or pass in era5.py as cache_dir parameter)
# execute run_accfs on HPC
# execute run_cocip on HPC

In [ ]:
# Import the results
from pathlib import Path

dffs = []
dfos = []
for day in range(8):
    path = Path(f"{path_to_data}/processed/day{day+1}")

    dff = pd.concat(
        (pd.read_parquet(f) for f in path.glob("*filed*.parquet")),
        ignore_index=True
    )
    dffs.append(dff)

    dfo = pd.concat(
        (pd.read_parquet(f) for f in path.glob("*optimised*.parquet")),
        ignore_index=True
    )
    dfos.append(dfo)

dffs = pd.concat(dffs, ignore_index=True)
dfos = pd.concat(dfos, ignore_index=True)

dffs.to_parquet(f"{path_to_data}/processed/filed_trajectories_before_metrics.parquet")
dfos.to_parquet(f"{path_to_data}/processed/optimised_trajectories_before_metrics.parquet")

In [ ]:
# Assign metrics
from pycontrails import Fleet

dff = pd.read_parquet(f"{path_to_data}/processed/filed_trajectories_before_metrics.parquet")
fleetf = Fleet(data=dff)
print(f"Fleet contains {fleetf.n_flights} flights")

dfo = pd.read_parquet(f"{path_to_data}/processed/optimised_trajectories_before_metrics.parquet")
fleeto = Fleet(data=dfo)
print(f"Fleet contains {fleeto.n_flights} flights")

In [ ]:
# Choose aCCF version
scaling = {
    "vanmanen_2019": {'CiC': 1, 'O3': 1, 'CH4': 1, 'H2O': 1},
    "yin_2023": {"CiC": 1, "O3": 1.97, "CH4": 2.03, "H2O": 1.92},
    "matthes_2023": {"CiC": 3, "O3": 11, "CH4": 35, "H2O": 3},
}
ver = "matthes_2023"

# Select aCCF mask
# [0, 1000] to keep them all and apply filtering while plotting; use [150, 350] for figure D1
accf_bounds = [0, 1000]  # [0, 1000], [150, 350]

# Choose efficacy
efficacy = {
    "dahlmann_2025": {"CiC": 0.21, "O3": 1.05, "CH4": 1.04, "H2O": 1},
    "none": {"CiC": 1, "O3": 1, "CH4": 1, "H2O": 1},
}
eff = "dahlmann_2025"

# Choose target metric
target = "EAGWP100"  # EAGWP100, ATR100, EAGWP20

In [ ]:
from cane.metrics.metrics import compute_metrics_for_fleet

fleetf = compute_metrics_for_fleet(
    fleet=fleetf,
    accf_version=ver,
    accf_scaling=scaling[ver],
    accf_bounds=accf_bounds,
    efficacy=efficacy[eff],
    target=target
)

fleeto = compute_metrics_for_fleet(
    fleet=fleeto,
    accf_version=ver,
    accf_scaling=scaling[ver],
    accf_bounds=accf_bounds,
    efficacy=efficacy[eff],
    target=target
)

In [ ]:
# Save results
fleetf.dataframe.to_parquet(f"{path_to_data}/processed/filed_trajectories_{target}.parquet")
fleeto.dataframe.to_parquet(f"{path_to_data}/processed/optimised_trajectories_{target}.parquet")